## Holiday Date  API Import to raw_files

In [0]:
%python
import requests
import json
import time

START_YEAR = 1990
END_YEAR = 2030
COUNTRY = "NZ"
VOLUME_PATH = "/Volumes/techsolve/ticket_bronze/raw_file/Input/"

all_holidays = []
failed_years = []

for year in range(START_YEAR, END_YEAR + 1):
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{COUNTRY}"
    try:
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            if data:  
                all_holidays.extend(data)
                print(f"{year}: {len(data)} holidays retrieved")
            else:
                print(f"{year}: no data returned (empty)")
                failed_years.append(year)
        else:
            print(f"{year}: request failed with status {response.status_code}")
            failed_years.append(year)
            
    except requests.exceptions.RequestException as e:
        print(f"{year}: request error - {e}")
        failed_years.append(year)
    
    time.sleep(0.2) 

print(f"\nTotal holidays collected: {len(all_holidays)}")
print(f"Years with no data: {failed_years}")

output_path = f"{VOLUME_PATH}/nz_public_holidays_1990_2030.json"
with open(output_path, "w") as f:
    json.dump(all_holidays, f, indent=2)

print(f"\nSaved to: {output_path}")


## Holiday Bronze delta_table

In [0]:
%python
from pyspark.sql import functions as F

df_holidays = spark.read.option("multiline", "true").json(
    "/Volumes/techsolve/ticket_bronze/raw_file/Input/nz_public_holidays_1990_2030.json"
)

df_holidays = df_holidays.withColumn("date", F.to_date("date", "yyyy-MM-dd"))
df_trimmed = df_holidays.select("date", "global", "name")
df_collapsed = (
    df_trimmed
    .groupBy("date")
    .agg(
        F.max("global").alias("global"),
        F.concat_ws(", ", F.collect_list("name")).alias("name")   # combine event names, comma-separated
    )
    .orderBy("date")
)

df_collapsed.write.format("delta").mode("overwrite").saveAsTable(
    "techsolve.ticket_bronze.holiday_lookup"
)

display(spark.table("techsolve.ticket_bronze.holiday_lookup"))

## Ticket_Bronze_Category

In [0]:
%sql
CREATE OR REPLACE TABLE techsolve.ticket_bronze.category_lookup (
    ticket_category STRING,
    cleaned_sub_category STRING,
    cleaned_category STRING
);

INSERT INTO techsolve.ticket_bronze.category_lookup (ticket_category,cleaned_sub_category, cleaned_category) VALUES
    ('Account Suspension', 'Account Suspension','Account Management'),
    ('Acct Suspension', 'Account Suspension', 'Account Management'),
    ('account_suspension', 'Account Suspension', 'Account Management'),

    ('BUG', 'Bug Report', 'Technical Issues'),
    ('Bug Report', 'Bug Report', 'Technical Issues'),
    ('Bug report', 'Bug Report' , 'Technical Issues'),
    ('bug_report', 'Bug Report', 'Technical Issues'),

    ('Data Sync', 'Data Sync Issue', 'Technical Issues'),
    ('Data Sync Issue', 'Data Sync Issue', 'Technical Issues'),
    ('data_sync', 'Data Sync Issue', 'Technical Issues'),

    ('Feature Req', 'Feature Request', 'Technical Issues'),
    ('Feature Request', 'Feature Request', 'Technical Issues'),
    ('feature_request', 'Feature Request', 'Technical Issues'),

    ('LOGIN ISSUE', 'Login Issue', 'Account Management'),
    ('Login Issue', 'Login Issue', 'Account Management'),
    ('Login issue', 'Login Issue', 'Account Management'),
    ('login_issue', 'Login Issue', 'Account Management'),

    ('Payment Problem', 'Payment Problem', 'Billing & Payments'),
    ('Payment problem', 'Payment Problem', 'Billing & Payments'),
    ('payment_problem', 'Payment Problem', 'Billing & Payments'),

    ('Perf Issue', 'Performance Issue', 'Technical Issues'),
    ('Performance Issue', 'Performance Issue', 'Technical Issues'),
    ('performance_issue', 'Performance Issue', 'Technical Issues'),

    ('Refund Req', 'Refund Request', 'Billing & Payments'),
    ('Refund Request', 'Refund Request', 'Billing & Payments'),
    ('refund_request', 'Refund Request', 'Billing & Payments'),

    ('Security', 'Security Concern','Security'),
    ('Security Concern', 'Security Concern','Security'),
    ('security_concern', 'Security Concern','Security'),

    ('Subscription Cancel', 'Subscription Cancellation','Billing & Payments'),
    ('Subscription Cancellation', 'Subscription Cancellation','Billing & Payments'),
    ('sub_cancellation', 'Subscription Cancellation','Billing & Payments');